# LLMPrice — Complete Feature Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TechyNilesh/LLMPrice/blob/main/notebooks/llmprice_demo.ipynb)

This notebook demonstrates all features of the **llmprice-kit** library — a fast, offline-first LLM pricing lookup for 2500+ models across all major providers.

## 1. Installation

In [1]:
!pip install -q llmprice-kit
print("Successfully installed llmprice-kit")

## 2. Initialize LLMPrice

The library ships with bundled pricing data — works offline with zero latency.

In [2]:
from llmprice import LLMPrice

# Default: uses bundled data (offline, instant)
lp = LLMPrice()

print(lp)
print(f"Total models: {lp.total_models}")

LLMPrice(models=2597, auto_update=False)
Total models: 2597


## 3. Get Model Pricing — Python Object

Returns a `ModelPrice` dataclass with all pricing and capability info.

In [3]:
# Get pricing as a Python object
model = lp.get("gpt-4o")

print(f"Model:           {model.name}")
print(f"Provider:        {model.provider}")
print(f"Input cost:      ${model.input_cost_per_1m} / 1M tokens")
print(f"Output cost:     ${model.output_cost_per_1m} / 1M tokens")
print(f"Max input:       {model.max_input_tokens:,} tokens")
print(f"Max output:      {model.max_output_tokens:,} tokens")
print(f"Vision:          {model.supports_vision}")
print(f"Function calls:  {model.supports_function_calling}")

Model:           gpt-4o
Provider:        openai
Input cost:      $2.5 / 1M tokens
Output cost:     $10.0 / 1M tokens
Max input:       128,000 tokens
Max output:      16,384 tokens
Vision:          True
Function calls:  True


## 4. Get Model Pricing — JSON Dict

Returns a JSON-serializable dictionary, useful for APIs and data pipelines.

In [4]:
import json

# Get pricing as a JSON-serializable dict
data = lp.get_json("gpt-4o")
print(json.dumps(data, indent=2))

{
  "name": "gpt-4o",
  "provider": "openai",
  "mode": "chat",
  "input_cost_per_1m": 2.5,
  "output_cost_per_1m": 10.0,
  "cache_read_cost_per_1m": 1.25,
  "cache_write_cost_per_1m": 0,
  "max_input_tokens": 128000,
  "max_output_tokens": 16384,
  "supports_vision": true,
  "supports_function_calling": true,
  "supports_reasoning": false,
  "supports_audio_input": false,
  "supports_audio_output": false,
  "supports_web_search": false,
  "supports_prompt_caching": true,
  "supports_response_schema": true,
  "deprecation_date": null
}


## 5. Compare Models Side by Side

Compare pricing and context windows across different providers.

In [5]:
# Compare popular models from different providers
models = lp.compare(["gpt-4o", "claude-opus-4-20250514", "gemini/gemini-2.0-flash", "deepseek/deepseek-chat"])

for m in models:
    print(f"{m.name:40s} ${m.input_cost_per_1m:>8.2f} in / ${m.output_cost_per_1m:>8.2f} out  | {m.max_input_tokens:>10,} ctx")

gpt-4o                                   $    2.50 in / $   10.00 out  |    128,000 ctx
claude-opus-4-20250514                   $   15.00 in / $   75.00 out  |    200,000 ctx
gemini/gemini-2.0-flash                  $    0.10 in / $    0.40 out  |  1,048,576 ctx
deepseek/deepseek-chat                   $    0.28 in / $    0.42 out  |    131,072 ctx


## 6. Compare as JSON

Get comparison results as a list of JSON dicts.

In [6]:
# Compare budget models — returns list of dicts
result = lp.compare_json(["gpt-4o-mini", "claude-haiku-4-5"])
print(json.dumps(result, indent=2))

[
  {
    "name": "gpt-4o-mini",
    "provider": "openai",
    "mode": "chat",
    "input_cost_per_1m": 0.15,
    "output_cost_per_1m": 0.6,
    "cache_read_cost_per_1m": 0.075,
    "cache_write_cost_per_1m": 0,
    "max_input_tokens": 128000,
    "max_output_tokens": 16384,
    "supports_vision": true,
    "supports_function_calling": true,
    "supports_reasoning": false,
    "supports_audio_input": false,
    "supports_audio_output": false,
    "supports_web_search": false,
    "supports_prompt_caching": true,
    "supports_response_schema": true,
    "deprecation_date": null
  },
  {
    "name": "claude-haiku-4-5",
    "provider": "anthropic",
    "mode": "chat",
    "input_cost_per_1m": 1.0,
    "output_cost_per_1m": 5.0,
    "cache_read_cost_per_1m": 0.09999999999999999,
    "cache_write_cost_per_1m": 1.25,
    "max_input_tokens": 200000,
    "max_output_tokens": 64000,
    "supports_vision": true,
    "supports_function_calling": true,
    "supports_reasoning": true,
    "suppor

## 7. Search Models by Capabilities

Find models that match specific requirements — vision, reasoning, price limits, context size.

In [7]:
# Find cheap vision models (input cost < $2 per 1M tokens)
results = lp.search(supports_vision=True, max_input_price=2.0, mode="chat")

print("Cheap vision models (input < $2/1M):")
for m in results[:10]:
    print(f"{m.name:50s} ${m.input_cost_per_1m:>6.2f} [{m.provider}]")
print(f"\nTotal found: {len(results)}")

Cheap vision models (input < $2/1M):
amazon.nova-lite-v1:0                              $  0.06 [bedrock_converse]
amazon.nova-2-lite-v1:0                            $  0.30 [bedrock_converse]
apac.amazon.nova-2-lite-v1:0                       $  0.33 [bedrock_converse]
eu.amazon.nova-2-lite-v1:0                         $  0.33 [bedrock_converse]
us.amazon.nova-2-lite-v1:0                         $  0.33 [bedrock_converse]
amazon.nova-pro-v1:0                               $  0.80 [bedrock_converse]
anthropic.claude-haiku-4-5-20251001-v1:0           $  1.00 [bedrock_converse]
anthropic.claude-haiku-4-5@20251001                $  1.00 [bedrock_converse]
anthropic.claude-3-haiku-20240307-v1:0             $  0.25 [bedrock]
apac.amazon.nova-lite-v1:0                         $  0.06 [bedrock_converse]

Total found: 347


In [8]:
# Find all Anthropic models with reasoning capability
results = lp.search(supports_reasoning=True, provider="anthropic")

print("Anthropic reasoning models:")
for m in results:
    print(f"{m.name:50s} ${m.input_cost_per_1m:>6.2f} / ${m.output_cost_per_1m:>6.2f}")

Anthropic reasoning models:
claude-haiku-4-5-20251001                          $  1.00 / $  5.00
claude-haiku-4-5                                   $  1.00 / $  5.00
claude-3-7-sonnet-20250219                         $  3.00 / $ 15.00
claude-4-opus-20250514                             $ 15.00 / $ 75.00
claude-4-sonnet-20250514                           $  3.00 / $ 15.00
claude-sonnet-4-5                                  $  3.00 / $ 15.00
claude-sonnet-4-5-20250929                         $  3.00 / $ 15.00
claude-sonnet-4-6                                  $  3.00 / $ 15.00
claude-opus-4-1                                    $ 15.00 / $ 75.00
claude-opus-4-1-20250805                           $ 15.00 / $ 75.00
claude-opus-4-20250514                             $ 15.00 / $ 75.00
claude-opus-4-5-20251101                           $  5.00 / $ 25.00
claude-opus-4-5                                    $  5.00 / $ 25.00
claude-opus-4-6                                    $  5.00 / $ 25.00
claude

## 8. Browse by Provider

Get all models from a specific provider.

In [9]:
# Get all OpenAI models
openai_models = lp.by_provider("openai")

print(f"OpenAI models: {len(openai_models)}")
for m in openai_models[:8]:
    print(f"  {m.name:40s} ${m.input_cost_per_1m:>8.2f} / ${m.output_cost_per_1m:>8.2f}")

OpenAI models: 203
  1024-x-1024/dall-e-2                     $    0.00 / $    0.00
  256-x-256/dall-e-2                       $    0.00 / $    0.00
  512-x-512/dall-e-2                       $    0.00 / $    0.00
  chatgpt-4o-latest                        $    5.00 / $   15.00
  gpt-4o-transcribe-diarize                $    2.50 / $   10.00
  codex-mini-latest                        $    1.50 / $    6.00
  dall-e-2                                 $    0.00 / $    0.00
  dall-e-3                                 $    0.00 / $    0.00


## 9. List All Providers

See every provider available in the dataset.

In [10]:
# List all available providers
providers = lp.providers()

print(f"Total providers: {len(providers)}")
print(", ".join(providers[:20]))

Total providers: 107
ai21, aiml, aleph_alpha, amazon_nova, anthropic, anyscale, assemblyai, aws_polly, azure, azure_ai, azure_text, bedrock, bedrock_converse, bedrock_mantle, black_forest_labs, cerebras, chatgpt, cloudflare, codestral, cohere


## 10. List All Models

In [11]:
# List all model names
all_models = lp.all_models()

print(f"Total models: {len(all_models)}")
print(f"First 10: {all_models[:10]}")

Total models: 2597
First 10: ['1024-x-1024/50-steps/bedrock/amazon.nova-canvas-v1:0', '1024-x-1024/50-steps/stability.stable-diffusion-xl-v1', '1024-x-1024/dall-e-2', '1024-x-1024/max-steps/stability.stable-diffusion-xl-v1', '256-x-256/dall-e-2', '512-x-512/50-steps/stability.stable-diffusion-xl-v0', '512-x-512/dall-e-2', '512-x-512/max-steps/stability.stable-diffusion-xl-v0', 'ai21.j2-mid-v1', 'ai21.j2-ultra-v1']


## 11. Check Data Freshness

See how old the local pricing data is.

In [12]:
# Check how fresh the data is
print(lp.data_age())

Data is 2h old.


## 12. Auto-Update Mode

Enable auto-update to fetch fresh data when the local copy is older than 1 day.

In [13]:
# Auto-update mode: fetches latest data if local copy is > 1 day old
lp_auto = LLMPrice(auto_update=True)
print(lp_auto)

LLMPrice(models=2597, auto_update=True)


In [14]:
# Or manually force a refresh
lp.update()
print(f"Data refreshed. Models: {lp.total_models}")

Data refreshed. Models: 2597


## 13. Access Raw LiteLLM Data

Each `ModelPrice` object holds the original LiteLLM data in the `.raw` field for advanced use cases.

In [15]:
# Access the raw LiteLLM data for a model
model = lp.get("gpt-4o")

print(f"Raw fields for {model.name}:")
for key, val in list(model.raw.items())[:5]:
    print(f"  {key}: {val}")

Raw fields for gpt-4o:
  cache_read_input_token_cost: 1.25e-06
  cache_read_input_token_cost_priority: 2.125e-06
  input_cost_per_token: 2.5e-06
  input_cost_per_token_batches: 1.25e-06
  input_cost_per_token_priority: 4.25e-06


---

That's it! For more info, check the [GitHub repo](https://github.com/TechyNilesh/LLMPrice) or install with:

```bash
pip install llmprice-kit
```